# Steam Trajectory — Data Exploration

Loading the fronkongames/steam-games-dataset **JSON** export (switched from CSV after validation revealed row-level misalignment — see notes in `kaggle_loader.py`). This notebook confirms the JSON loads cleanly, then runs cohort selection.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("/Users/pmacias/Dropbox/steamproject")
print(os.getcwd())

/Users/pmacias/Dropbox/steamproject


In [2]:
from steam_trajectory.ingest.kaggle_loader import KaggleLoader

# NOTE: adjust filename below to match whatever the JSON actually
# downloaded as (likely games.json — confirm in data/raw/ first)
loader = KaggleLoader("data/raw/games.json")
df = loader.df
print(df.shape)
df.head()

(135043, 9)


,AppID,Name,Release date,Developers,Publishers,Price,total_reviews,review_score_percent,Genres
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",NaN,NaN,0.00,0,NaN,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",minori,MangaGamer,5.24,255,98.823529,Adventure
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",Somer Games,8floor,4.99,24,87.500000,Casual
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",유진게임즈,유진게임즈,8.99,0,NaN,"Casual, Indie, Simulation"
4,3631080,Maze Quest VR,"Apr 24, 2025",Reality Expanded LLC,Reality Expanded LLC,4.99,0,NaN,"Action, Early Access"


## Validation
Confirming the JSON loaded cleanly before trusting `select_cohort()`'s output. No CSV-style alignment issues are possible here (JSON fields are explicitly keyed, not positional), but worth still checking for data-quality issues (missing dates, bad AppIDs, etc.) independent of the format itself.

In [3]:
import pandas as pd

# 1. Do release dates parse cleanly?
release_dates = pd.to_datetime(df["Release date"], errors="coerce")
print("Unparseable release dates:", release_dates.isna().sum(), "of", len(df))

Unparseable release dates: 0 of 135043


In [4]:
# 2. Is AppID unique and always an integer? Our whole schema's
# primary key depends on this being true.
print("Duplicate AppIDs:", df["AppID"].duplicated().sum())
print("AppID dtype:", df["AppID"].dtype)

Duplicate AppIDs: 0
AppID dtype: int64


In [5]:
# 3. Is Price always numeric and non-negative?
print("Price dtype:", df["Price"].dtype)
print("Negative prices:", (df["Price"] < 0).sum())
print("Missing prices:", df["Price"].isna().sum())

Price dtype: float64
Negative prices: 0
Missing prices: 0


In [6]:
# 4. Spot-check: do name/date/price/genre all look like they
# plausibly belong to the same game, across a random sample?
df.sample(10, random_state=1)[["Name", "Release date", "Price", "Genres"]]

,Name,Release date,Price,Genres
109723,Blacksmith Simulator,"Nov 28, 2025",5.99,"Adventure, Casual, Indie, Simulation"
26780,"Ethyrial, Echoes of Yore Playtest","Jan 24, 2025",0.00,NaN
78923,ETERNAL EXILE: BENEATH THE DARKNESS,"Aug 20, 2025",15.99,"Action, Indie"
51423,Where is my car? Playtest,"Sep 30, 2025",0.00,NaN
125316,PengPong: Prologue,"Feb 20, 2026",0.00,"Action, Casual, Indie, Free To Play"
88912,Stagehand Survival Simulator Playtest,"Feb 13, 2023",0.00,NaN
90366,Smash and Bash Monsters: Prologue,"Jul 22, 2024",0.00,"Action, Casual, Indie, RPG, Free To Play"
127770,Chronicles: Immortal Sect,"Mar 28, 2026",0.00,"Adventure, Massively Multiplayer, RPG, Free To..."
45862,Video World,"Jan 21, 2021",0.00,Indie
33603,Prince Of Wallachia,"Apr 28, 2021",0.59,"Action, Adventure, Indie"


In [7]:
# 5. total_reviews / review_score_percent sanity check
print(df[["total_reviews", "review_score_percent"]].describe())

       total_reviews  review_score_percent
count   1.350430e+05          82979.000000
mean    1.102677e+03             75.824456
std     3.110025e+04             23.868394
min     0.000000e+00              0.000000
25%     0.000000e+00             64.964021
50%     4.000000e+00             81.818182
75%     3.900000e+01             94.444444
max     8.815087e+06            100.000000


## Cohort selection
If everything above looks clean, select the actual research cohort: 2019–2022 releases with 500+ reviews.

In [10]:
candidates = loader.select_cohort(
    release_start="2019-01-01",
    release_end="2022-12-31",
    min_reviews=500,
    sample_size=260,  # buffer above 200, since some candidates will
                      # fail SteamCharts scraping in notebook 01
)
print(candidates.shape)
candidates[["AppID", "Name", "Release date", "total_reviews"]].head(10)

(260, 9)


,AppID,Name,Release date,total_reviews
0,1030210,Torchlight III,"Oct 13, 2020",11230
1,852090,Penko Park,"Oct 23, 2020",578
2,1100420,Praetorians - HD Remaster,"Jan 24, 2020",985
3,1248130,Farming Simulator 22,"Nov 21, 2021",84512
4,1808780,祖玛少女/Zuma Girls,"May 1, 2022",684
5,1100990,Aimbeast,"May 13, 2020",1978
6,1734320,Brutal Orchestra,"Dec 17, 2021",1800
7,1367590,Tormented Souls,"Aug 26, 2021",6089
8,1218250,We Went Back,"Apr 3, 2020",4627
9,1043390,FlowScape,"Aug 15, 2019",1313


In [11]:
candidates.to_csv("candidate_appids.csv", index=False)
print(f"Saved {len(candidates)} candidates to candidate_appids.csv")

Saved 260 candidates to candidate_appids.csv
